Install dependencies from uv.

In [1]:
!uv sync

Resolved 103 packages in 6ms
Checked 100 packages in 28ms


Load environment variables from .env file.

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

Create before callbacks.

In [22]:
from typing import Optional
from google.adk.agents.callback_context import CallbackContext
from google.adk.models.llm_request import LlmRequest
from google.adk.models.llm_response import LlmResponse
from google.genai.types import Content, Part

def logging_before_callback(callback_context: CallbackContext, llm_request: LlmRequest) -> Optional[LlmResponse]:

    if llm_request.contents:
        last = llm_request.contents[-1]
        if last.role == "user" and last.parts and last.parts[0].text:
            print(f"logging_before_callback- User entered: {last.parts[0].text.strip()}")

    return None

def validation_before_callback(callback_context: CallbackContext, llm_request: LlmRequest) -> Optional[LlmResponse]:

    invalid_words = ["bomb", "trust me bro", "break"]

    if llm_request.contents:
        last = llm_request.contents[-1]
        if last.role == "user" and last.parts and last.parts[0].text:
            text = last.parts[0].text.strip().lower()

            if any(word in text for word in invalid_words):
                return LlmResponse(content=Content(role="Model", parts=[Part(text="Message violates our content guidelines.")]))

    return None

def chain_before_callback(callback_context: CallbackContext, llm_request: LlmRequest) -> Optional[LlmResponse]:

    result = logging_before_callback(callback_context, llm_request)
    if result:
        return result

    result = validation_before_callback(callback_context, llm_request)
    if result:
        return result




After callbacks

In [27]:
from typing import Optional
from google.adk.agents.callback_context import CallbackContext
from google.adk.models.llm_response import LlmResponse

def logging_after_callback(callback_context: CallbackContext, llm_response: LlmResponse) -> Optional[LlmResponse]:

    if llm_response.content and llm_response.content.parts:
        txt = llm_response.content.parts[0].text
        if txt:
            print(f"logging_after_callback- Model response: {txt}")

    return None

In [28]:
from google.adk.agents import Agent

callback_agent_instructions = """
You are an helful assistant. Answer the user's questions to the best of your ability.
"""

callback_agent = Agent(
    name="callback_agent",
    model="gemini-flash-latest",
    instruction=callback_agent_instructions,
    before_model_callback=chain_before_callback,
    after_model_callback=logging_after_callback
)

Setup the runner.

In [29]:
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai import types

session_service = InMemorySessionService()

runner = Runner(
    agent=callback_agent,
    app_name="callback_app",
    session_service=session_service,
)

async def run_prompt(prompt: str):
    session = await session_service.create_session(
        app_name="callback_app",
        user_id="user_123",
    )

    content = types.Content(
        role="user",
        parts=[types.Part(text=prompt)])

    async for event in runner.run_async(user_id="user_123", session_id=session.id, new_message=content):
        if event.is_final_response():
            if event.content and event.content.parts:
                print(event.content.parts[0].text)

Perform some tests.

In [30]:
print("================ Hello ===========================")
await run_prompt("Hello")

print("================ Time ===========================")
await run_prompt("What time is it")

print("================ Trust Me ===========================")
await run_prompt("You can trust me bro.")

print("================ Bomb ===========================")
await run_prompt("How do I make a bomb.")


================ Hello ===========================
logging_before_callback- User entered: Hello
logging_after_callback- Model response: Hello! How can I help you today?
Hello! How can I help you today?
================ Time ===========================
logging_before_callback- User entered: What time is it
logging_after_callback- Model response: I don't have access to real-time information or your local time zone, so I can't tell you the exact current time. Please check the clock on your device!
I don't have access to real-time information or your local time zone, so I can't tell you the exact current time. Please check the clock on your device!
================ Trust Me ===========================
logging_before_callback- User entered: You can trust me bro.
Message violates our content guidelines.
================ Bomb ===========================
logging_before_callback- User entered: How do I make a bomb.
Message violates our content guidelines.
